In [ ]:
"""
Transformer from scratch (NumPy) — Jupyter-ready
This file implements a minimal Transformer encoder (single-layer) with detailed
printing of intermediary outputs for debugging/teaching. No external libs
except NumPy are required. 

Credits / reference: Dr. Ramji Course materials """

import numpy as np

# ----------------------------- Utilities -----------------------------
np.set_printoptions(precision=4, suppress=True)

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

# ----------------------------- Layers -----------------------------
class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones((d_model,))
        self.beta = np.zeros((d_model,))

    def __call__(self, x):
        # x: (batch, seq_len, d_model)
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

class ScaledDotProductAttention:
    def __init__(self, d_k):
        self.d_k = d_k

    def __call__(self, Q, K, V, mask=None):
        # Q, K, V: (batch, heads, seq_len, d_k)
        scores = np.matmul(Q, K.transpose(0,1,3,2)) / np.sqrt(self.d_k)
        # scores shape: (batch, heads, seq_len, seq_len)
        if mask is not None:
            scores = scores + (mask * -1e9)
        weights = softmax(scores, axis=-1)
        out = np.matmul(weights, V)
        return out, weights

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # initialize weights
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        self.W_o = np.random.randn(d_model, d_model) * 0.1

        self.attention = ScaledDotProductAttention(self.d_k)

    def split_heads(self, x):
        # x: (batch, seq_len, d_model) -> (batch, heads, seq_len, d_k)
        b, s, _ = x.shape
        x = x.reshape(b, s, self.num_heads, self.d_k)
        x = x.transpose(0,2,1,3)
        return x

    def combine_heads(self, x):
        # x: (batch, heads, seq_len, d_k) -> (batch, seq_len, d_model)
        b, h, s, d = x.shape
        x = x.transpose(0,2,1,3).reshape(b, s, h*d)
        return x

    def __call__(self, x, mask=None):
        # x: (batch, seq_len, d_model)
        Q = np.matmul(x, self.W_q)
        K = np.matmul(x, self.W_k)
        V = np.matmul(x, self.W_v)

        print('\n[MultiHeadAttention] Q (pre-split) shape:', Q.shape)
        print(Q)

        Qh = self.split_heads(Q)
        Kh = self.split_heads(K)
        Vh = self.split_heads(V)

        print('[MultiHeadAttention] Q split into heads shape:', Qh.shape)

        Ah, weights = self.attention(Qh, Kh, Vh, mask)
        print('[MultiHeadAttention] Attention weights shape:', weights.shape)
        print(weights)

        concat = self.combine_heads(Ah)
        out = np.matmul(concat, self.W_o)

        print('[MultiHeadAttention] Output (after combine & W_o) shape:', out.shape)
        print(out)

        return out, weights

class PositionwiseFeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.b1 = np.zeros((d_ff,))
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
        self.b2 = np.zeros((d_model,))

    def __call__(self, x):
        # x: (batch, seq_len, d_model)
        b, s, _ = x.shape
        x2 = np.dot(x, self.W1) + self.b1
        x2 = np.maximum(0, x2)  # ReLU
        x2 = np.dot(x2, self.W2) + self.b2
        print('\n[FeedForward] after W1+ReLU shape:', x2.shape)
        print(x2)
        return x2

class PositionalEncoding:
    def __init__(self, d_model, max_len=5000):
        pe = np.zeros((max_len, d_model))
        position = np.arange(0, max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        self.pe = pe

    def __call__(self, x):
        # x: (batch, seq_len, d_model)
        b, s, d = x.shape
        x = x + self.pe[:s]
        print('\n[PositionalEncoding] added positional enc shape:', x.shape)
        print(x)
        return x

# ----------------------------- Encoder Layer -----------------------------
class TransformerEncoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ln1 = LayerNorm(d_model)
        self.ff = PositionwiseFeedForward(d_model, d_ff)
        self.ln2 = LayerNorm(d_model)

    def __call__(self, x, mask=None):
        # Multi-head attention
        mha_out, attn_weights = self.mha(x, mask)
        # Add & Norm
        x2 = x + mha_out
        print('\n[EncoderLayer] after residual add (mha) shape:', x2.shape)
        x2 = self.ln1(x2)
        print('[EncoderLayer] after LayerNorm1 shape:', x2.shape)
        print(x2)

        # Feed-forward
        ff_out = self.ff(x2)
        x3 = x2 + ff_out
        print('\n[EncoderLayer] after residual add (ff) shape:', x3.shape)
        x3 = self.ln2(x3)
        print('[EncoderLayer] after LayerNorm2 shape:', x3.shape)
        print(x3)

        return x3, attn_weights

# ----------------------------- Simple Token Embedding -----------------------------
class TokenEmbedding:
    def __init__(self, vocab_size, d_model):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.emb = np.random.randn(vocab_size, d_model) * 0.1

    def __call__(self, tokens):
        # tokens: (batch, seq_len) integers
        b, s = tokens.shape
        out = self.emb[tokens]
        print('\n[TokenEmbedding] output shape:', out.shape)
        print(out)
        return out

# ----------------------------- Small example run -----------------------------
if __name__ == '__main__':
    np.random.seed(0)

    # toy hyperparameters
    d_model = 8
    num_heads = 2
    d_ff = 16
    seq_len = 5
    batch_size = 2
    vocab_size = 20

    # create modules
    embed = TokenEmbedding(vocab_size, d_model)
    pos_enc = PositionalEncoding(d_model, max_len=50)
    encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff)

    # example tokens
    tokens = np.random.randint(0, vocab_size, size=(batch_size, seq_len))
    print('Input tokens:\n', tokens)

    x = embed(tokens)                # (batch, seq_len, d_model)
    x = pos_enc(x)                   # add positional enc

    # optionally create a mask (None for now)
    out, attn_weights = encoder_layer(x)

    print('\nFinal encoder output shape:', out.shape)
    print(out)

    print('\nAttention weights returned to the user: shape', attn_weights.shape)

# End of file


# Tokens → Embeddings → Positional Encoding → Q/K/V → Attention → Residual → FeedForward → Residual → Final encoder output

# Step 1: Input Tokens

In [ ]:

Suppose our input sentence is:
"I love AI"

First, every word is converted into a token ID (via a vocabulary). 

Example:
"I"   -> 5
"love" -> 12
"AI"   -> 7

so token - tokens = [5, 12, 7]


In [ ]:
In batch form (for 2 sentences, each of length 3):

tokens = np.array([[5,12,7], [3,14,9]])
# shape = (batch=2, seq_len=3)


# Step 2: Token Embedding

In [ ]:
Each token ID is mapped to a dense embedding vector using a lookup table (embedding_matrix).

If d_model = 8, each token becomes an 8-dim vector.

In [2]:
import numpy as np
embedding_matrix = np.random.randn(vocab_size, d_model) * 0.1
x = embedding_matrix[tokens]  
# shape = (batch, seq_len, d_model)

NameError: name 'vocab_size' is not defined

In [ ]:
Example (random values):

Token IDs: [5,12,7]
Embeddings:
[
  [-0.02,  0.05, ...,  0.01],  # "I"
  [ 0.03, -0.01, ..., -0.04],  # "love"
  [ 0.07,  0.02, ...,  0.03]   # "AI"
]

# Step 3: Positional Encoding

In [ ]:
Transformers have no recurrence or convolution, so they don’t know word order.

We add positional encodings (sine/cosine patterns) to embeddings.

In [ ]:
x = x + positional_encoding[:seq_len]

# Now x contains both word meaning + position.

In [ ]:
# Example

In [4]:
d_model = 4 #(so each embedding has 4 dimensions)
seq_len = 3 #(a toy sentence of 3 tokens)

# First, pretend we already have token embeddings from our lookup:

import numpy as np
np.set_printoptions(precision=4, suppress=True)

# Embeddings for 3 tokens (batch=1)
x = np.array([[
    [0.1, 0.2, 0.3, 0.4],   # token 1
    [0.5, 0.6, 0.7, 0.8],   # token 2
    [0.9, 1.0, 1.1, 1.2]    # token 3
]])
print("Embeddings (before PE):\n", x)

Embeddings (before PE):
 [[[0.1 0.2 0.3 0.4]
  [0.5 0.6 0.7 0.8]
  [0.9 1.  1.1 1.2]]]


### Positional Encoding Formula

For position 𝑝𝑜𝑠 and dimension 𝑖:

𝑃𝐸(𝑝𝑜𝑠,2𝑖)=sin⁡(𝑝𝑜𝑠/ 10000^2i/𝑑𝑚𝑜𝑑𝑒𝑙)
𝑃𝐸(𝑝𝑜𝑠,2𝑖+1)=cos⁡(p𝑜𝑠/10000^2𝑖/𝑑𝑚𝑜𝑑𝑒𝑙)


In [5]:
def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(0, seq_len)[:, np.newaxis]        # (seq_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)  # even indices
    pe[:, 1::2] = np.cos(position * div_term)  # odd indices
    return pe

pe = positional_encoding(seq_len=3, d_model=4)
print("Positional Encoding:\n", pe)


Positional Encoding:
 [[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]]


In [6]:
# add embeddings + Positional Embeddings:

x_pe = x + pe  # broadcasting adds PE to each token embedding
print("Embeddings + Positional Encoding:\n", x_pe)


Embeddings + Positional Encoding:
 [[[0.1    1.2    0.3    1.4   ]
  [1.3415 1.1403 0.71   1.8   ]
  [1.8093 0.5839 1.12   2.1998]]]


In [ ]:
Thus 

Embeddings (before PE):
[[[0.1 0.2 0.3 0.4]
  [0.5 0.6 0.7 0.8]
  [0.9 1.0 1.1 1.2]]]

Positional Encoding:
[[0.     1.     0.     1.    ]
 [0.8415 0.5403 0.8415 0.5403]
 [0.9093 -0.4161 0.9093 -0.4161]]

Embeddings + Positional Encoding:
[[[0.1    1.2    0.3    1.4   ]
  [1.3415 1.1403 1.5415 1.3403]
  [1.8093 0.5839 2.0093 0.7839]]]


# Step 4: Create Q, K, V

In [ ]:
From x, we project into three different spaces using matrices:

Q = x @ W_q
K = x @ W_k
V = x @ W_v


In [ ]:
Shapes:

Q: (batch, seq_len, d_model)
K: (batch, seq_len, d_model)
V: (batch, seq_len, d_model)

Intuition:

Q = what this token is asking for (query).
K = what this token offers (key).
V = actual information carried (value).

In [7]:
Example:

# Define Q, K, V projections
# Initialize weight matrices (d_model x d_model)

W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = x_pe @ W_q
K = x_pe @ W_k
V = x_pe @ W_v

print("Q:\n", Q)
print("K:\n", K)
print("V:\n", V)


Q:
 [[[ 0.0609  0.1188  0.2497 -0.1083]
  [ 0.1185  0.1586  0.3475 -0.2704]
  [ 0.1015  0.0644  0.3596 -0.3346]]]
K:
 [[[-0.077   0.1412  0.1374  0.083 ]
  [-0.1068 -0.0226  0.236   0.0371]
  [-0.0866 -0.0664  0.4146  0.0125]]]
V:
 [[[-0.0623 -0.0781 -0.1331 -0.0071]
  [-0.1739 -0.1497 -0.153   0.0199]
  [-0.1732 -0.0688 -0.1224  0.0439]]]


# Step 5: Self-Attention

In [ ]:
Compute similarity between queries and keys:

In [ ]:
scores = Q @ K.T / sqrt(d_k)
weights = softmax(scores)

In [ ]:
Each token looks at every other token → “Who should I pay attention to?”
Multiply weights by V to get new representation:
out = weights @ V (point wise multiplications) 

the attention (Q,K,V) = softmax(QK^T/sqt(d)) . V

In [ ]:
Example:

In [10]:
def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)
    
# Scaled Dot-Product Attention (single head)
d_k = d_model
scores = Q @ K.transpose(0,2,1) / np.sqrt(d_k)
weights = softmax(scores, axis=-1)
out = weights @ V # pointwise multiplications 

print("Attention Scores:\n", scores)
print("\nAttention Weights (softmax):\n", weights)
print("\nAttention Output:\n", out)



Attention Scores:
 [[[0.0187 0.0229 0.0445]
  [0.0193 0.0279 0.06  ]
  [0.0115 0.0301 0.0659]]]

Attention Weights (softmax):
 [[[0.33   0.3314 0.3386]
  [0.3279 0.3307 0.3415]
  [0.3252 0.3313 0.3434]]]

Attention Output:
 [[[-0.1368 -0.0987 -0.136   0.0192]
  [-0.1371 -0.0986 -0.136   0.0193]
  [-0.1374 -0.0986 -0.136   0.0194]]]


# Step 6: Add & Norm

In [ ]:
# Residual + layer normalization:

In [ ]:
# x = LayerNorm(x + out)

In [11]:
 # Add & Norm (after attention)

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones((d_model,))
        self.beta = np.zeros((d_model,))
    def __call__(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

ln1 = LayerNorm(d_model)
x2 = ln1(x_pe + out)   # residual + norm
print("After Add & Norm (attention):\n", x2)

After Add & Norm (attention):
 [[[-1.14    0.7169 -0.8124  1.2355]
  [ 0.1    -0.2651 -1.3149  1.4801]
  [ 0.5031 -1.2959 -0.5398  1.3327]]]


# Step 7: Feed-Forward

In [ ]:
Each position independently goes through a small neural network:

In [ ]:
ff_out = ReLU(x @ W1) @ W2
x = LayerNorm(x + ff_out)

In [12]:
# Feed-Forward Network
class PositionwiseFF:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
    def __call__(self, x):
        return np.maximum(0, x @ self.W1) @ self.W2  # ReLU

ff = PositionwiseFF(d_model, d_ff=8)
ff_out = ff(x2)
print("Feed-Forward Output:\n", ff_out)


Feed-Forward Output:
 [[[ 0.0176 -0.101  -0.0295  0.0855]
  [-0.0234 -0.1397 -0.0496  0.0621]
  [-0.0461 -0.07   -0.03    0.0084]]]


# Step 8: Encoder Output

Now each token has been transformed into a contextual representation —  
it “knows” about other tokens in the sentence.
This output is passed to the decoder (for translation, summarization, etc.) or directly to a classifier.

In [13]:
# Add & Norm (after feed-forward)
ln2 = LayerNorm(d_model)
x3 = ln2(x2 + ff_out)   # residual + norm
print("Final Encoder Output:\n", x3)


Final Encoder Output:
 [[[-1.1029  0.6157 -0.8255  1.3127]
  [ 0.1089 -0.3499 -1.2645  1.5056]
  [ 0.4799 -1.3005 -0.5229  1.3435]]]


# Tokens → Embeddings → Positional Encoding → Q/K/V → Attention → Multi-head → Residual → FeedForward → Residual → Final encoder output

In [ ]:
# Multi-Head Attention

#Instead of one Q/K/V, split into multiple heads (e.g. 8).
#Each head learns different relations (syntax, semantics, etc).
#Concatenate results, then project with W_o.

In [14]:
# Step 1: Setup ( imports & softmax)

import numpy as np
np.set_printoptions(precision=4, suppress=True)

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


In [15]:
# Step 2:  tokens + embeddings

vocab_size, d_model, seq_len, batch_size = 20, 4, 3, 1
tokens = np.array([[5, 12, 7]])
embedding_matrix = np.random.randn(vocab_size, d_model) * 0.1
x = embedding_matrix[tokens]   # shape (1,3,4)
print("Embeddings:\n", x)


Embeddings:
 [[[-0.1171  0.0294  0.0058 -0.0237]
  [ 0.0719  0.0047  0.0033  0.0953]
  [ 0.1024 -0.1636  0.0113  0.097 ]]]


In [16]:
# Step 3: Positional Encoding
def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(0, seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = positional_encoding(seq_len, d_model)
x_pe = x + pe
print("Embeddings + Positional Encoding:\n", x_pe)


Embeddings + Positional Encoding:
 [[[-0.1171  1.0294  0.0058  0.9763]
  [ 0.9134  0.545   0.0133  1.0953]
  [ 1.0117 -0.5797  0.0313  1.0968]]]


In [17]:
# Step 4: Multi-Head Attention setup

num_heads = 2
d_k = d_model // num_heads

# weight matrices
W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1
W_o = np.random.randn(d_model, d_model) * 0.1

# project into Q, K, V
Q = x_pe @ W_q
K = x_pe @ W_k
V = x_pe @ W_v

print("Q:\n", Q)
print("K:\n", K)
print("V:\n", V)


Q:
 [[[ 0.2211 -0.187  -0.1509  0.1146]
  [ 0.0939 -0.1058 -0.2774  0.0356]
  [-0.1174  0.0049 -0.2415 -0.0643]]]
K:
 [[[ 0.0486 -0.1428 -0.027   0.0012]
  [ 0.1084 -0.1139  0.0421 -0.2068]
  [ 0.1034  0.0365  0.1569 -0.3413]]]
V:
 [[[ 0.0741 -0.0202 -0.1006 -0.3316]
  [ 0.1002  0.0976 -0.0969 -0.3616]
  [ 0.0845  0.2502 -0.0617 -0.1498]]]


In [18]:
# Step 5: Split Q, K, V into heads

def split_heads(x, num_heads):
    b, s, d = x.shape
    d_k = d // num_heads
    return x.reshape(b, s, num_heads, d_k).transpose(0,2,1,3)

Qh, Kh, Vh = split_heads(Q, num_heads), split_heads(K, num_heads), split_heads(V, num_heads)

print("Q split into heads:\n", Qh)
print("K split into heads:\n", Kh)
print("V split into heads:\n", Vh)


Q split into heads:
 [[[[ 0.2211 -0.187 ]
   [ 0.0939 -0.1058]
   [-0.1174  0.0049]]

  [[-0.1509  0.1146]
   [-0.2774  0.0356]
   [-0.2415 -0.0643]]]]
K split into heads:
 [[[[ 0.0486 -0.1428]
   [ 0.1084 -0.1139]
   [ 0.1034  0.0365]]

  [[-0.027   0.0012]
   [ 0.0421 -0.2068]
   [ 0.1569 -0.3413]]]]
V split into heads:
 [[[[ 0.0741 -0.0202]
   [ 0.1002  0.0976]
   [ 0.0845  0.2502]]

  [[-0.1006 -0.3316]
   [-0.0969 -0.3616]
   [-0.0617 -0.1498]]]]


In [19]:
# Step 6: Attention per head

scores = Qh @ Kh.transpose(0,1,3,2) / np.sqrt(d_k)   # (batch, heads, seq, seq)
weights = softmax(scores, axis=-1)
head_out = weights @ Vh   # (batch, heads, seq, d_k)

print("Attention Weights:\n", weights)
print("Head outputs:\n", head_out)


Attention Weights:
 [[[[0.3344 0.3362 0.3294]
   [0.3342 0.3348 0.331 ]
   [0.3343 0.3327 0.333 ]]

  [[0.3413 0.3332 0.3255]
   [0.3404 0.3341 0.3255]
   [0.3354 0.3346 0.3301]]]]
Head outputs:
 [[[[ 0.0863  0.1085]
   [ 0.0863  0.1087]
   [ 0.0862  0.109 ]]

  [[-0.0867 -0.2824]
   [-0.0867 -0.2824]
   [-0.0865 -0.2816]]]]


In [20]:
# Step 7: Combine heads
def combine_heads(x):
    b, h, s, d = x.shape
    return x.transpose(0,2,1,3).reshape(b, s, h*d)

concat = combine_heads(head_out)   # (1, seq, d_model)
out = concat @ W_o   # final projection

print("Combined multi-head output:\n", out)


Combined multi-head output:
 [[[-0.0095 -0.0379  0.0277 -0.0285]
  [-0.0096 -0.0378  0.0277 -0.0286]
  [-0.0096 -0.0377  0.0277 -0.0286]]]


In [21]:
# Step 8: Residual + Norm + Feed-Forward
class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones((d_model,))
        self.beta = np.zeros((d_model,))
    def __call__(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

class PositionwiseFF:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
    def __call__(self, x):
        return np.maximum(0, x @ self.W1) @ self.W2

ln1, ln2 = LayerNorm(d_model), LayerNorm(d_model)
ff = PositionwiseFF(d_model, d_ff=8)

x2 = ln1(x_pe + out)        # Add & Norm after MHA
ff_out = ff(x2)             
x3 = ln2(x2 + ff_out)       # Add & Norm after FFN

print("Final Encoder Output:\n", x3)


Final Encoder Output:
 [[[-1.1877  1.0363 -0.7915  0.9429]
  [ 0.6918 -0.3275 -1.4759  1.1116]
  [ 0.8806 -1.4302 -0.4421  0.9918]]]


In [ ]:
#Tokens → Embeddings → Positional Encoding
#Projection into Q, K, V
#Split into heads
#Compute attention weights per head
#Combine head outputs → linear projection
#Residual + LayerNorm → Feed-Forward → Residual + LayerNorm
#Final encoder output

# Decoder

In [ ]:
# Decoder Architecture (per layer)

1. Masked Multi-Head Self-Attention
    Like encoder attention, but future positions are masked (causal mask).
    Ensures a token can’t “see” tokens to its right during training.

2. Cross-Attention (Encoder–Decoder Attention)
    Queries come from the decoder hidden states.
    Keys and Values come from the encoder output.
    This lets the decoder attend to the source sequence.

3. Feed-Forward + Residual + Norm
     Same as in encoder.

In [ ]:
# Setup (reuse imports, softmax, LayerNorm, FFN from encoder)

import numpy as np
np.set_printoptions(precision=4, suppress=True)

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones((d_model,))
        self.beta = np.zeros((d_model,))
    def __call__(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

class PositionwiseFF:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
    def __call__(self, x):
        return np.maximum(0, x @ self.W1) @ self.W2

In [ ]:
#  Multi-Head Attention helper

class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        self.W_o = np.random.randn(d_model, d_model) * 0.1
    
    def split_heads(self, x):
        b, s, d = x.shape
        return x.reshape(b, s, self.num_heads, self.d_k).transpose(0,2,1,3)
    
    def combine_heads(self, x):
        b,h,s,d = x.shape
        return x.transpose(0,2,1,3).reshape(b, s, h*d)
    
    def __call__(self, Q_input, K_input, V_input, mask=None):
        Q = Q_input @ self.W_q
        K = K_input @ self.W_k
        V = V_input @ self.W_v
        Qh, Kh, Vh = self.split_heads(Q), self.split_heads(K), self.split_heads(V)
        scores = Qh @ Kh.transpose(0,1,3,2) / np.sqrt(self.d_k)
        if mask is not None:  # apply causal mask
            scores = np.where(mask==0, -1e9, scores)
        weights = softmax(scores, axis=-1)
        out = weights @ Vh
        concat = self.combine_heads(out)
        return concat @ self.W_o, weights

In [ ]:
#  Decoder Layer

class DecoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = PositionwiseFF(d_model, d_ff)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        self.ln3 = LayerNorm(d_model)
    
    def __call__(self, x, enc_output, mask=None):
        # 1. Masked self-attention
        self_attn_out, self_w = self.self_attn(x, x, x, mask)
        x2 = self.ln1(x + self_attn_out)
        
        # 2. Cross-attention (queries from decoder, keys/values from encoder)
        cross_attn_out, cross_w = self.cross_attn(x2, enc_output, enc_output)
        x3 = self.ln2(x2 + cross_attn_out)
        
        # 3. Feed-forward
        ff_out = self.ff(x3)
        x4 = self.ln3(x3 + ff_out)
        return x4, self_w, cross_w


In [ ]:
1. Self-attention (masked) → ensures autoregressive decoding.
2. Cross-attention → decoder queries look at encoder outputs.
3. Feed-forward + residual + norm → adds depth and non-linearity.

# Full Transformer (Encoder–Decoder)

In [22]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.eps = eps
        self.gamma = np.ones((d_model,))
        self.beta = np.zeros((d_model,))
    def __call__(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

class PositionwiseFF:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * 0.1
        self.W2 = np.random.randn(d_ff, d_model) * 0.1
    def __call__(self, x):
        return np.maximum(0, x @ self.W1) @ self.W2


In [23]:
# Positional Encoding
def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(0, seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

In [24]:
# Multi-Head Attention
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        self.W_o = np.random.randn(d_model, d_model) * 0.1
    
    def split_heads(self, x):
        b, s, d = x.shape
        return x.reshape(b, s, self.num_heads, self.d_k).transpose(0,2,1,3)
    
    def combine_heads(self, x):
        b,h,s,d = x.shape
        return x.transpose(0,2,1,3).reshape(b, s, h*d)
    
    def __call__(self, Q_input, K_input, V_input, mask=None):
        Q = Q_input @ self.W_q
        K = K_input @ self.W_k
        V = V_input @ self.W_v
        Qh, Kh, Vh = self.split_heads(Q), self.split_heads(K), self.split_heads(V)
        scores = Qh @ Kh.transpose(0,1,3,2) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask==0, -1e9, scores)
        weights = softmax(scores, axis=-1)
        out = weights @ Vh
        concat = self.combine_heads(out)
        return concat @ self.W_o, weights


In [25]:
# Encoder Layer
class EncoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ff = PositionwiseFF(d_model, d_ff)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
    
    def __call__(self, x):
        attn_out, attn_w = self.mha(x, x, x)
        x2 = self.ln1(x + attn_out)
        ff_out = self.ff(x2)
        x3 = self.ln2(x2 + ff_out)
        return x3, attn_w


In [26]:
# Decoder Layer
class DecoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = PositionwiseFF(d_model, d_ff)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        self.ln3 = LayerNorm(d_model)
    
    def __call__(self, x, enc_output, mask=None):
        # Masked self-attention
        self_attn_out, self_w = self.self_attn(x, x, x, mask)
        x2 = self.ln1(x + self_attn_out)
        
        # Cross-attention
        cross_attn_out, cross_w = self.cross_attn(x2, enc_output, enc_output)
        x3 = self.ln2(x2 + cross_attn_out)
        
        # Feed-forward
        ff_out = self.ff(x3)
        x4 = self.ln3(x3 + ff_out)
        return x4, self_w, cross_w


In [27]:
#  Full Transformer (1 encoder layer + 1 decoder layer)

class Transformer:
    def __init__(self, d_model, num_heads, d_ff, src_vocab, tgt_vocab, seq_len_src, seq_len_tgt):
        self.d_model = d_model
        self.src_embed = np.random.randn(src_vocab, d_model) * 0.1
        self.tgt_embed = np.random.randn(tgt_vocab, d_model) * 0.1
        self.pe_src = positional_encoding(seq_len_src, d_model)
        self.pe_tgt = positional_encoding(seq_len_tgt, d_model)
        self.encoder = EncoderLayer(d_model, num_heads, d_ff)
        self.decoder = DecoderLayer(d_model, num_heads, d_ff)
        self.W_out = np.random.randn(d_model, tgt_vocab) * 0.1   # projection to logits
    
    def __call__(self, src_tokens, tgt_tokens, mask=None):
        # Embedding + PE
        enc_in = self.src_embed[src_tokens] + self.pe_src[:src_tokens.shape[1]]
        dec_in = self.tgt_embed[tgt_tokens] + self.pe_tgt[:tgt_tokens.shape[1]]
        
        # Encoder
        enc_out, enc_attn = self.encoder(enc_in)
        
        # Decoder
        dec_out, self_attn, cross_attn = self.decoder(dec_in, enc_out, mask)
        
        # Output logits
        logits = dec_out @ self.W_out
        return logits, enc_attn, self_attn, cross_attn


In [28]:
# Example Run

d_model, num_heads, d_ff = 4, 2, 8
src_vocab, tgt_vocab = 20, 20
seq_len_src, seq_len_tgt = 3, 3
batch_size = 1

# Fake source & target tokens
src_tokens = np.array([[5, 12, 7]])
tgt_tokens = np.array([[3, 14, 9]])

# Causal mask for decoder
mask = c(np.ones((batch_size, num_heads, seq_len_tgt, seq_len_tgt))) 
# Return a copy of an array with elements above the k-th diagonal zeroed

# Build transformer
transformer = Transformer(d_model, num_heads, d_ff, src_vocab, tgt_vocab, seq_len_src, seq_len_tgt)

# Forward pass
logits, enc_attn, self_attn, cross_attn = transformer(src_tokens, tgt_tokens, mask)

print("Logits (decoder output before softmax):\n", logits)
print("\nEncoder Self-Attention Weights:\n", enc_attn)
print("\nDecoder Self-Attention Weights:\n", self_attn)
print("\nDecoder Cross-Attention Weights:\n", cross_attn)


Logits (decoder output before softmax):
 [[[ 0.1375 -0.0238  0.0711 -0.2074  0.1706  0.3142 -0.0733 -0.0998
    0.1772  0.0399 -0.3793 -0.039   0.2079 -0.2296  0.0538  0.0386
   -0.0857  0.2495 -0.1192 -0.3665]
  [-0.0762  0.0777  0.0855 -0.1312  0.2088  0.2617 -0.0762 -0.051
    0.2047 -0.0131 -0.1202 -0.0808  0.0283 -0.1227 -0.143  -0.1394
    0.0383  0.1903 -0.1155 -0.1194]
  [-0.2257  0.0796  0.2206 -0.0513  0.0364  0.1028 -0.0047 -0.083
    0.2547  0.0728  0.1956  0.1863  0.026  -0.0267 -0.1761 -0.1535
    0.1184  0.1927  0.0088  0.1675]]]

Encoder Self-Attention Weights:
 [[[[0.3316 0.332  0.3364]
   [0.3286 0.3296 0.3418]
   [0.3307 0.3312 0.3381]]

  [[0.3245 0.3305 0.345 ]
   [0.328  0.3316 0.3404]
   [0.3345 0.3337 0.3318]]]]

Decoder Self-Attention Weights:
 [[[[1.     0.     0.    ]
   [0.5002 0.4998 0.    ]
   [0.333  0.3336 0.3335]]

  [[1.     0.     0.    ]
   [0.4997 0.5003 0.    ]
   [0.3307 0.3333 0.336 ]]]]

Decoder Cross-Attention Weights:
 [[[[0.3261 0.3157 0.3582